# MODEL A: Balanced 

 **balanced** performance across all classes.

**Strategy**:
- **Loss**: Adaptive Focal Loss with class-specific gamma (higher for weak classes)
- **Scheduler**: SGDR (Cosine Annealing with Warm Restarts)
- **Data**: Balanced pseudo-labels (1500 per class, 2:1 ratio)
- **Focus**: Overall balance, slight bias toward weak classes

**Class-Specific Configuration**:
```
                 Gamma  Weight  Focus
Economic         1.5    0.70    Low (already strong)
Human Impact     2.0    1.00    Medium
None             2.0    1.05    Medium
Conflict         2.5    1.30    High (weak)
Moral Value      3.5    1.60    MAX (weakest)
Powerlessness    3.5    1.65    MAX (weakest)
```


In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import pickle
import warnings
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("✅ Imports loaded")
print(f"Device: {device}")

✅ Imports loaded
Device: cuda


## 1) Load Data

In [2]:
print("Loading augmented training data...")
augmented = pd.read_csv('data/augmented_training_data.csv', keep_default_na=False, na_values=[''])
print(f"✅ Augmented data: {len(augmented)} samples")

# Split into train/val/test
real_train = augmented[augmented['split'] == 'train'].copy()
real_val = augmented[augmented['split'] == 'validation'].copy()
real_test = augmented[augmented['split'] == 'test'].copy()

print(f"  Train: {len(real_train)}")
print(f"  Val: {len(real_val)}")
print(f"  Test: {len(real_test)}")

print("\nLoading pseudo-labeled data (STRICT: conf>0.92, entropy<0.3)...")
pseudo = pd.read_csv('data/pseudo_labeled_chunks.csv', keep_default_na=False, na_values=[''])
pseudo_train = pseudo.copy()
pseudo_train['split'] = 'train'
pseudo_train.rename(columns={'pseudo_label': 'frame_label'}, inplace=True)
print(f"✅ Pseudo-labeled: {len(pseudo_train)} samples")
print(f"  Per-class:")
print(pseudo_train['frame_label'].value_counts().sort_index())

# Add sample weights: real=1.0, pseudo=0.2
real_train['sample_weight'] = 1.0
pseudo_train['sample_weight'] = 0.2

combined_train = pd.concat([real_train, pseudo_train], ignore_index=True)
print(f"\n✅ Combined train: {len(combined_train)}")
print(f"  Real: {len(real_train)} (1.0) | Pseudo: {len(pseudo_train)} (0.2)")
print(f"  Ratio: {len(pseudo_train)/len(real_train):.2f}:1")

Loading augmented training data...
✅ Augmented data: 5637 samples
  Train: 4606
  Val: 683
  Test: 348

Loading pseudo-labeled data (STRICT: conf>0.92, entropy<0.3)...
✅ Pseudo-labeled: 9000 samples
  Per-class:
frame_label
Conflict         1500
Economic         1500
Human Impact     1500
Moral Value      1500
None             1500
Powerlessness    1500
Name: count, dtype: int64

✅ Combined train: 13606
  Real: 4606 (1.0) | Pseudo: 9000 (0.2)
  Ratio: 1.95:1


## 2) Encode Labels & Tokenize

In [3]:
with open('data/roberta_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
labels = list(label_encoder.classes_)
num_labels = len(labels)
print(f"Labels: {labels}")

# Encode
combined_train['label'] = label_encoder.transform(combined_train['frame_label'])
real_val['label'] = label_encoder.transform(real_val['frame_label'])
real_test['label'] = label_encoder.transform(real_test['frame_label'])

# Class weights (moderate, balanced focus)
class_weights = torch.FloatTensor([
    1.30,  # Conflict
    0.70,  # Economic (de-emphasize)
    1.00,  # Human Impact
    1.60,  # Moral Value
    1.05,  # None
    1.65   # Powerlessness
])
print(f"✅ Class weights: {class_weights.tolist()}")

# Tokenizer
tokenizer = RobertaTokenizer.from_pretrained('models/roberta-dapt-dynamic')

def tokenize_function(examples):
    return tokenizer(examples['chunk_text'], truncation=True, max_length=384, padding='max_length')

# Datasets
train_dataset = Dataset.from_pandas(combined_train[['chunk_text','label','sample_weight']])
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
train_dataset.set_format('torch')

val_dataset = Dataset.from_pandas(real_val[['chunk_text','label']])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
val_dataset.set_format('torch')

test_dataset = Dataset.from_pandas(real_test[['chunk_text','label']])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
test_dataset.set_format('torch')

print(f"✅ Datasets: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

Labels: ['Conflict', 'Economic', 'Human Impact', 'Moral Value', 'None', 'Powerlessness']
✅ Class weights: [1.2999999523162842, 0.699999988079071, 1.0, 1.600000023841858, 1.0499999523162842, 1.649999976158142]


Map:   0%|          | 0/13606 [00:00<?, ? examples/s]

Map:   0%|          | 0/683 [00:00<?, ? examples/s]

Map:   0%|          | 0/348 [00:00<?, ? examples/s]

✅ Datasets: Train=13606, Val=683, Test=348


## 3) Adaptive Focal Loss

In [4]:
class AdaptiveFocalLoss(nn.Module):
    """Focal Loss with class-specific gamma"""
    def __init__(self, class_weights, gamma_per_class, device):
        super().__init__()
        self.class_weights = class_weights.to(device)
        self.gamma_per_class = gamma_per_class.to(device)
    
    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce)
        gamma = self.gamma_per_class[labels]
        focal_term = (1 - pt) ** gamma
        weights = self.class_weights[labels]
        loss = focal_term * weights * ce
        return loss.mean()

gamma_per_class = torch.FloatTensor([
    2.5,  # Conflict
    1.5,  # Economic (low)
    2.0,  # Human Impact
    3.5,  # Moral Value (MAX)
    2.0,  # None
    3.5   # Powerlessness (MAX)
])

print("✅ Adaptive Focal Loss")
print(f"  Gamma: {gamma_per_class.tolist()}")

✅ Adaptive Focal Loss
  Gamma: [2.5, 1.5, 2.0, 3.5, 2.0, 3.5]


## 4) Custom Collator & Trainer

In [5]:
from dataclasses import dataclass
from typing import Any, Dict, List
from transformers import DataCollatorWithPadding

@dataclass
class DataCollatorWithSampleWeight(DataCollatorWithPadding):
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        sample_weights = None
        if 'sample_weight' in features[0]:
            sample_weights = torch.FloatTensor([f.pop('sample_weight') for f in features])
        batch = super().__call__(features)
        if sample_weights is not None:
            batch['sample_weight'] = sample_weights
        return batch

class AdaptiveFocalTrainer(Trainer):
    def __init__(self, *args, focal_loss_fn=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = focal_loss_fn
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        sample_weight = inputs.pop('sample_weight', None)
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        ce = F.cross_entropy(logits.view(-1, self.model.config.num_labels), labels.view(-1), reduction='none')
        pt = torch.exp(-ce)
        gamma = self.focal_loss_fn.gamma_per_class[labels.view(-1)]
        focal_term = (1 - pt) ** gamma
        weights = self.focal_loss_fn.class_weights[labels.view(-1)]
        per_sample_loss = focal_term * weights * ce
        
        if sample_weight is not None:
            loss = (per_sample_loss * sample_weight).mean()
        else:
            loss = per_sample_loss.mean()
        
        return (loss, outputs) if return_outputs else loss

print("✅ Custom trainer")

✅ Custom trainer


## 5) Load Model

In [6]:
print("Loading RoBERTa from models/roberta-dapt-dynamic...")
model = RobertaForSequenceClassification.from_pretrained(
    'models/roberta-dapt-dynamic',
    num_labels=num_labels,
    attention_probs_dropout_prob=0.05,
    hidden_dropout_prob=0.10,
    ignore_mismatched_sizes=True
)
model = model.to(device)
print("✅ Model loaded")

Loading RoBERTa from models/roberta-dapt-dynamic...
✅ Model loaded


## 6) Optimizer & SGDR

In [7]:
encoder_params = []
classifier_params = []

for name, param in model.named_parameters():
    if 'classifier' in name:
        classifier_params.append(param)
    else:
        encoder_params.append(param)

encoder_lr = 3e-5
classifier_lr = 2e-3

optimizer = AdamW([
    {'params': encoder_params, 'lr': encoder_lr},
    {'params': classifier_params, 'lr': classifier_lr}
], weight_decay=0.05)

scheduler = CosineAnnealingWarmRestarts(
    optimizer, T_0=300, T_mult=2, eta_min=1e-6
)

print("✅ SGDR scheduler")
print(f"  Encoder LR: {encoder_lr}")
print(f"  Classifier LR: {classifier_lr}")

✅ SGDR scheduler
  Encoder LR: 3e-05
  Classifier LR: 0.002


## 7) Metrics

In [8]:
def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels_np, preds)
    f1_macro = f1_score(labels_np, preds, average='macro')
    f1_weighted = f1_score(labels_np, preds, average='weighted')
    _, _, f1_per_class, _ = precision_recall_fscore_support(
        labels_np, preds, average=None, zero_division=0
    )
    metrics = {'accuracy': acc, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted}
    for i, label in enumerate(labels):
        metrics[f'f1_{label}'] = f1_per_class[i]
    return metrics

print("✅ Metrics function")

✅ Metrics function


## 8) Training Arguments

In [9]:
training_args = TrainingArguments(
    output_dir='models/roberta-ensemble-model-A',
    num_train_epochs=12,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    weight_decay=0.05,
    max_grad_norm=5.0,
    eval_strategy='steps',
    eval_steps=50,  # Evaluate every 50 steps!
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    label_smoothing_factor=0.0,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    logging_steps=50,
    report_to='none',
    seed=42
)

print("✅ Training args")

✅ Training args


## 9) Train

In [10]:
focal_loss_fn = AdaptiveFocalLoss(class_weights, gamma_per_class, device)
data_collator = DataCollatorWithSampleWeight(tokenizer=tokenizer, padding=True)

trainer = AdaptiveFocalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    data_collator=data_collator,
    optimizers=(optimizer, scheduler),
    focal_loss_fn=focal_loss_fn
)

print("\n" + "="*80)
print("🚀 MODEL A: BALANCED SPECIALIST")
print("="*80)
print("Loss: Adaptive Focal (gamma 1.5-3.5)")
print("Scheduler: SGDR (T_0=300, restarts)")
print("Target: F1=0.72, balanced classes")
print("="*80)

train_result = trainer.train()
print("\n✅ Training complete!")


🚀 MODEL A: BALANCED SPECIALIST
Loss: Adaptive Focal (gamma 1.5-3.5)
Scheduler: SGDR (T_0=300, restarts)
Target: F1=0.72, balanced classes


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Conflict,F1 Economic,F1 Human impact,F1 Moral value,F1 None,F1 Powerlessness
50,0.261700,1.107606,0.653001,0.654529,0.656100,0.611872,0.785714,0.704348,0.582375,0.625767,0.617100
100,0.083800,1.277648,0.636896,0.642124,0.642079,0.571429,0.696517,0.696970,0.617544,0.710900,0.559387
150,0.073400,1.414645,0.663250,0.661228,0.661836,0.636015,0.801498,0.652174,0.582960,0.702128,0.592593
200,0.052300,1.031220,0.680820,0.681012,0.681606,0.631579,0.815126,0.707424,0.620408,0.737374,0.574163
250,0.026100,1.058389,0.683748,0.682750,0.683221,0.640693,0.800000,0.709091,0.643478,0.740741,0.562500
300,0.018500,1.025027,0.685212,0.686544,0.686985,0.614679,0.813278,0.711712,0.633929,0.748768,0.596899
350,0.055300,1.724493,0.629575,0.628534,0.628689,0.560000,0.765766,0.664407,0.514768,0.725490,0.540773
400,0.122800,1.564213,0.603221,0.605155,0.604516,0.502793,0.637681,0.651982,0.601695,0.706977,0.529801
450,0.086200,1.664026,0.686676,0.681771,0.682336,0.623762,0.787879,0.736842,0.672000,0.739336,0.530806
500,0.063000,1.264609,0.701318,0.698224,0.698790,0.629442,0.808000,0.763158,0.643478,0.756522,0.588745



✅ Training complete!


## 10) Evaluate & Save

In [11]:
print("\nValidation...")
val_results = trainer.evaluate(eval_dataset=val_dataset)
print(val_results)

print("\nTest...")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(test_results)

# Save
os.makedirs('models/roberta-ensemble-model-A', exist_ok=True)
trainer.save_model('models/roberta-ensemble-model-A')
tokenizer.save_pretrained('models/roberta-ensemble-model-A')

os.makedirs('results', exist_ok=True)
with open('results/model_A_metrics.json', 'w') as f:
    json.dump({
        'model': 'A - Balanced Specialist',
        'strategy': 'Adaptive Focal + SGDR',
        'gamma': gamma_per_class.tolist(),
        'weights': class_weights.tolist(),
        'val': val_results,
        'test': test_results,
        'time': train_result.metrics['train_runtime']
    }, f, indent=2)

print("\n" + "="*80)
print("MODEL A RESULTS:")
print(f"Val F1: {val_results['eval_f1_macro']:.4f}")
print(f"Test F1: {test_results['eval_f1_macro']:.4f}")
print("\nPer-Class F1:")
for label in labels:
    print(f"  {label:20s}: {test_results[f'eval_f1_{label}']:.4f}")
print("="*80)


Validation...


{'eval_loss': 1.2646089792251587, 'eval_accuracy': 0.7013177159590044, 'eval_f1_macro': 0.6982240179744855, 'eval_f1_weighted': 0.6987903125577899, 'eval_f1_Conflict': 0.6294416243654822, 'eval_f1_Economic': 0.808, 'eval_f1_Human Impact': 0.7631578947368421, 'eval_f1_Moral Value': 0.6434782608695652, 'eval_f1_None': 0.7565217391304347, 'eval_f1_Powerlessness': 0.5887445887445888, 'eval_runtime': 26.4463, 'eval_samples_per_second': 25.826, 'eval_steps_per_second': 0.832, 'epoch': 1.7619047619047619}

Test...
{'eval_loss': 1.1560614109039307, 'eval_accuracy': 0.6781609195402298, 'eval_f1_macro': 0.6783104289433265, 'eval_f1_weighted': 0.6783104289433264, 'eval_f1_Conflict': 0.6666666666666666, 'eval_f1_Economic': 0.7413793103448276, 'eval_f1_Human Impact': 0.6722689075630253, 'eval_f1_Moral Value': 0.6190476190476191, 'eval_f1_None': 0.7603305785123967, 'eval_f1_Powerlessness': 0.6101694915254238, 'eval_runtime': 13.496, 'eval_samples_per_second': 25.785, 'eval_steps_per_second': 0.815, 